# 03 · Persona LoRA-DPO

Use TRL DPOTrainer and PEFT for the provided persona preferences. The CLI restricts Trainer to one GPU to keep effective batch size 32.

Upload this notebook to Kaggle. Enable Internet; select **GPU T4 x2** for training/generation. The judge notebook needs benchmark model access, not a GPU. Enter your public GitHub URL below. No GitHub credentials are needed.


## 1. Configuration

Run All defaults to smoke checks only. After inspecting them, set `RUN_FULL=True`. Use a new run name if changing experimental settings; completed runs are skipped.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPO_URL = "https://github.com/0x0shephard/genai-pa2.git"
GIT_REF = "main"  # After the first run, use its printed commit SHA for all final runs.
REPO_DIR = Path("/kaggle/working/pa2-repo")
OUTPUT_ROOT = Path("/kaggle/working/outputs")
PREPARED_ROOT = Path("/kaggle/working/prepared")
RUN_SMOKE = True
RUN_FULL = False  # Inspect smoke output, then set True and rerun the configuration/full cells.

RUN_NAME = "persona"
RESUME_CHECKPOINT = None  # e.g. /kaggle/input/saved-persona/outputs/persona/trainer/checkpoint-20


## 2. Clone the chosen GitHub revision and install dependencies


In [ ]:
assert "YOUR_" not in REPO_URL, "Enter your public GitHub repository URL in the configuration cell."
assert REPO_URL.startswith("https://github.com/"), "Use a public GitHub HTTPS URL."
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    assert origin.rstrip("/").removesuffix(".git") == REPO_URL.rstrip("/").removesuffix(".git"), "Existing checkout belongs to another repository."
    assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True).strip(), "Checkout has local changes; preserve them before changing revisions."
subprocess.check_call(["git", "fetch", "origin", GIT_REF], cwd=REPO_DIR)
subprocess.check_call(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
os.environ["ALIGNMENT_GIT_REF"] = GIT_REF
os.environ["PYTHONPATH"] = str(REPO_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/kaggle/working/hf-cache"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREPARED_ROOT.mkdir(parents=True, exist_ok=True)
print("Git commit:", COMMIT)

def run(*args):
    subprocess.check_call([sys.executable, "-m", "alignment", *map(str, args)], cwd=REPO_DIR)

def need(path, label):
    path = Path(path)
    if not path.exists():
        print("Available attachments:", list(Path("/kaggle/input").glob("*/*")))
        raise FileNotFoundError(f"Attach the saved {label} output and update its configuration path: {path}")
    return path

def completed(path):
    summary = Path(path) / "summary.json"
    return summary.exists() and json.loads(summary.read_text()).get("completed", False)

def show_samples(path, count=5):
    for row in json.loads(Path(path).read_text())[:count]:
        print("PROMPT:", row["instruction"])
        print("RESPONSE:", row["output"], "\n")


In [ ]:
# Keep Kaggle's installed CUDA PyTorch. These requirements do not pin torch.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-persona.txt"), "--upgrade-strategy", "only-if-needed"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"])
subprocess.check_call(["nvidia-smi"])
subprocess.check_call([sys.executable, "-c", "import torch, transformers, datasets; print('torch', torch.__version__, 'transformers', transformers.__version__, 'datasets', datasets.__version__); print('GPU count:', torch.cuda.device_count()); assert torch.cuda.device_count() >= 2, 'Enable GPU T4 x2'"])
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "tests/test_public_adapter_smoke.py"], cwd=REPO_DIR)


## 3. Prepare the fixed split and smoke-test TRL


In [ ]:
run("prepare", "--track", "persona", "--output", PREPARED_ROOT)
if RUN_SMOKE:
    smoke_run = OUTPUT_ROOT / "smoke_persona"
    if not completed(smoke_run):
        run("persona", "--data", PREPARED_ROOT / "persona", "--output", smoke_run, "--max-steps", 2)
    print((smoke_run / "summary.json").read_text())


## 4. Full persona run

Rank 16, alpha 32, attention projections, dropout 0.05, LR 5e-5, three epochs. See the report template for the rationale to verify against actual results.


In [ ]:
if RUN_FULL:
    destination = OUTPUT_ROOT / RUN_NAME
    if not completed(destination):
        args = ["persona", "--data", PREPARED_ROOT / "persona", "--output", destination]
        if RESUME_CHECKPOINT:
            args += ["--resume", need(RESUME_CHECKPOINT, "TRL resume checkpoint")]
        run(*args)
    run("plot", "--run", destination, "--output", destination / "training.png", "--name", "Persona LoRA-DPO")
else:
    print("Smoke only. Set RUN_FULL=True for the final persona run.")


## Preserve outputs

Use **Save Version → Save & Run All** after setting `RUN_FULL=True`, or save completed interactive outputs with the appropriate Kaggle version option. Attach that saved notebook's output to the next stage using **Add Input → Notebook Output**. Confirm the files exist in the saved version before ending the session.

New artifacts are under `/kaggle/working/outputs` and `/kaggle/working/prepared`. Model files are large: do not duplicate them into zip archives unless disk permits. For interruption recovery, attach the prior saved output and configure its resume path. A file in a temporary session is preserved only when Kaggle saves the output.
